<a href="https://colab.research.google.com/github/hromeothando-lgtm/mineralprospect/blob/master/LSI_feature_engineering_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


# -----------------------------------------------------------------------------
# 1. DEPENDENCY INSTALLATION
# -----------------------------------------------------------------------------
print("[PHASE 0] Installing dependencies...")
!pip install geemap earthengine-api rasterio scipy numpy geedim --break-system-packages --quiet
print("[PHASE 0] Dependencies installed.")

# -----------------------------------------------------------------------------
# 2. IMPORTS
# -----------------------------------------------------------------------------
import os
import time
import shutil
import numpy as np
import rasterio
from scipy.ndimage import uniform_filter
import ee
import geemap
from google.colab import drive

# -----------------------------------------------------------------------------
# 3. ENVIRONMENT TUNING
# -----------------------------------------------------------------------------
os.environ["GEE_HTTP_MAX_CONNECTIONS"] = "5"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

# -----------------------------------------------------------------------------
# 4. CONFIGURATION
# -----------------------------------------------------------------------------
ASSET_PREFIX = "projects/mineral-prospectivity-zim/assets/tiles/feature_stack_tile_"

PATHFINDER_BANDS = [
    (5, "S2_IOI_nd"),
    (6, "S2_FII_nd"),
    (7, "S2_CI_nd"),
    (24, "L8_IOI_nd"),
    (25, "L8_FII_nd"),
    (26, "L8_CI_nd"),
]

WINDOW_SIZES = [3, 5, 7, 9, 11]
BUFFER_PX = 5
DE = 2.0

TEMP_DIR = "/content/temp_tiles"
OUTPUT_DIR = "/content/lsi_outputs"

for d in [TEMP_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# -----------------------------------------------------------------------------
# 5. GEE AUTHENTICATION
# -----------------------------------------------------------------------------
print("[PHASE 1] Authenticating with Earth Engine...")
try:
    ee.Initialize(project="mineral-prospectivity-zim")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="mineral-prospectivity-zim")
print("[PHASE 1] Earth Engine ready.")

# -----------------------------------------------------------------------------
# 6. MOUNT GOOGLE DRIVE
# -----------------------------------------------------------------------------
print("[PHASE 1] Mounting Google Drive...")
drive.mount("/content/drive", force_remount=True)
DRIVE_BACKUP_FOLDER = "/content/drive/MyDrive/LSI_Final_Outputs"
os.makedirs(DRIVE_BACKUP_FOLDER, exist_ok=True)
print("✅ [PHASE 1] Environment ready.")

# -----------------------------------------------------------------------------
# 7. LSI MATH — v3: NaN-AWARE WINDOWED DENSITY
# -----------------------------------------------------------------------------
def compute_windowed_density(data, window_sizes):
    """
    FIXED (v3): mask-aware windowed mean, equivalent to what GEE's
    reduceNeighborhood(mean) does natively. NaN pixels are excluded from
    each window's average instead of contaminating the whole window.

    Method: for each window size, compute (sum of valid values) / (count
    of valid values) per pixel, using uniform_filter as a fast box-sum
    engine on a NaN-zeroed copy of the data and a validity mask.
    """
    mask = np.isfinite(data)
    data0 = np.where(mask, data, 0.0)
    mask_f = mask.astype(np.float64)

    stacks = []
    for w in window_sizes:
        # uniform_filter computes a MEAN over the window; multiplying by
        # w*w converts it back to a SUM, which we need for the masked
        # ratio below.
        sum_vals = uniform_filter(data0, size=w, mode="constant", cval=0.0) * (w * w)
        count_vals = uniform_filter(mask_f, size=w, mode="constant", cval=0.0) * (w * w)

        with np.errstate(invalid="ignore", divide="ignore"):
            local_mean = sum_vals / count_vals

        # Only mark NaN where there was truly zero valid data in the
        # entire window (e.g. deep inside a large cloud gap) — not just
        # because one pixel nearby was masked.
        local_mean = np.where(count_vals > 0, local_mean, np.nan)
        stacks.append(local_mean)

    return np.stack(stacks, axis=0)

def compute_alpha_map(density_stack, window_sizes, De=2.0):
    """
    Robust log-log slope with strict NaN/Inf masking.
    Invalid pixels are filled with De (2.0) — this should now only
    happen for pixels with genuinely no valid data at any scale,
    not universally across the tile.
    """
    n_windows, h, w = density_stack.shape
    log_w = np.log(window_sizes)
    log_d = np.log(np.maximum(density_stack, 1e-9))

    y = log_d.reshape(n_windows, -1)
    valid_mask = np.all(np.isfinite(y), axis=0)

    if not np.any(valid_mask):
        return np.full((h, w), De, dtype=np.float32)

    x = log_w
    x_diff = x - np.mean(x)
    ss_xx = np.sum(x_diff ** 2)
    if ss_xx == 0:
        return np.full((h, w), De, dtype=np.float32)

    y_clean = y[:, valid_mask]
    y_mean = np.mean(y_clean, axis=0)
    ss_xy = np.sum(x_diff[:, None] * (y_clean - y_mean), axis=0)
    slope = ss_xy / ss_xx

    alpha = np.full((h, w), De, dtype=np.float64)
    alpha.flat[valid_mask] = slope
    return np.nan_to_num(alpha, nan=De, posinf=De, neginf=De).astype(np.float32)

# -----------------------------------------------------------------------------
# 8. SINGLE-TILE WORKER
# -----------------------------------------------------------------------------
def process_one_tile(tile_idx):
    asset_id = f"{ASSET_PREFIX}{tile_idx}"
    band_names = [b[1] for b in PATHFINDER_BANDS]

    print(f"[Tile {tile_idx}] [PHASE 2] Selecting bands: {band_names}")
    image = ee.Image(asset_id).select(band_names)
    region = image.geometry()

    local_raw = os.path.join(TEMP_DIR, f"tile_{tile_idx}.tif")

    print(f"[Tile {tile_idx}] [PHASE 2] Downloading 6-band subset (spatial tiling)...")
    geemap.download_ee_image(
        image,
        filename=local_raw,
        scale=30,
        region=region,
        crs="EPSG:4326",
        num_threads=4,
        max_tile_size=16,
        max_tile_dim=500,
    )

    if not os.path.exists(local_raw) or os.path.getsize(local_raw) < 1000:
        raise RuntimeError(f"Download failed for tile {tile_idx}")

    print(f"[Tile {tile_idx}] [PHASE 3] Download complete. Computing LSI maps...")

    try:
        with rasterio.open(local_raw) as src:
            base_profile = src.profile.copy()
            base_transform = src.transform

            for i, (_, band_name) in enumerate(PATHFINDER_BANDS):
                print(f"[Tile {tile_idx}] [PHASE 3]   band {i+1}/6: {band_name}...")
                band_data = src.read(i + 1).astype(np.float64)
                band_abs = np.abs(band_data)

                nan_frac = np.isnan(band_abs).mean()
                print(f"[Tile {tile_idx}] [PHASE 3]     NaN fraction in raw band: {nan_frac:.3f}")

                density_stack = compute_windowed_density(band_abs, WINDOW_SIZES)
                alpha_map = compute_alpha_map(density_stack, WINDOW_SIZES, De=DE)

                # Diagnostic: confirm this tile/band actually produced variance
                print(f"[Tile {tile_idx}] [PHASE 3]     alpha stdDev: {np.nanstd(alpha_map):.5f}, "
                      f"mean: {np.nanmean(alpha_map):.5f}")

                h, w = alpha_map.shape
                if h > 2 * BUFFER_PX and w > 2 * BUFFER_PX:
                    alpha_cropped = alpha_map[BUFFER_PX:h - BUFFER_PX, BUFFER_PX:w - BUFFER_PX]
                    new_transform = rasterio.transform.from_origin(
                        base_transform.c + BUFFER_PX * base_transform.a,
                        base_transform.f + BUFFER_PX * base_transform.e,
                        base_transform.a,
                        -base_transform.e,
                    )
                else:
                    alpha_cropped = alpha_map
                    new_transform = base_transform

                out_profile = base_profile.copy()
                out_profile.update({
                    "driver": "GTiff",
                    "dtype": "float32",
                    "count": 1,
                    "height": alpha_cropped.shape[0],
                    "width": alpha_cropped.shape[1],
                    "transform": new_transform,
                    "nodata": None,
                })

                out_path = os.path.join(OUTPUT_DIR, f"lsi_tile_{tile_idx}_{band_name}.tif")
                temp_path = out_path + ".tmp"

                with rasterio.open(temp_path, "w", **out_profile) as dst:
                    dst.write(alpha_cropped, 1)

                shutil.move(temp_path, out_path)
                print(f"[Tile {tile_idx}] [PHASE 3]   -> saved {out_path}")

        print(f"[Tile {tile_idx}] [PHASE 4] Deleting raw tile...")
        os.remove(local_raw)
        print(f"[Tile {tile_idx}] [PHASE 4] Complete. Raw tile deleted.\n")

    except Exception as e:
        if os.path.exists(local_raw):
            os.remove(local_raw)
        raise RuntimeError(f"Processing failed for tile {tile_idx}: {e}")

# -----------------------------------------------------------------------------
# 9. RESUME HELPER
# -----------------------------------------------------------------------------
def tile_is_done(tile_idx):
    for _, band_name in PATHFINDER_BANDS:
        f = os.path.join(OUTPUT_DIR, f"lsi_tile_{tile_idx}_{band_name}.tif")
        if not os.path.exists(f) or os.path.getsize(f) < 10000:
            return False
    return True

# -----------------------------------------------------------------------------
# 10. MAIN EXECUTION LOOP
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("[PHASE 5] STARTING NaN-AWARE LSI PIPELINE (45 TILES)")
print("=" * 60 + "\n")

total_start = time.time()
processed = 0
skipped = 0
failed_tiles = []

for idx in range(45):
    if tile_is_done(idx):
        print(f"[Tile {idx}] Already processed (valid size). Skipping.")
        skipped += 1
        continue

    try:
        start_tile = time.time()
        process_one_tile(idx)
        processed += 1
        print(f"   └─ Tile {idx} took {time.time() - start_tile:.1f}s")
    except Exception as e:
        print(f"[Tile {idx}] ❌ FAILED: {e}")
        failed_tiles.append(idx)

total_elapsed = time.time() - total_start

print("\n" + "=" * 60)
print("[PHASE 6] PIPELINE FINISHED")
print("=" * 60)
print(f"✅ Processed:  {processed} tiles")
print(f"⏭️  Skipped:   {skipped} tiles (already valid)")
print(f"❌ Failed:    {len(failed_tiles)} tiles → {failed_tiles if failed_tiles else 'None'}")
print(f"⏱️  Total time: {total_elapsed / 60:.1f} minutes")

# -----------------------------------------------------------------------------
# 11. BACKUP TO GOOGLE DRIVE
# -----------------------------------------------------------------------------
if processed > 0 or skipped > 0:
    print("\n[PHASE 7] 📤 Backing up LSI outputs to Google Drive (valid files only)...")
    count = 0
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith(".tif") and os.path.getsize(os.path.join(OUTPUT_DIR, f)) > 10000:
            shutil.copy2(
                os.path.join(OUTPUT_DIR, f),
                os.path.join(DRIVE_BACKUP_FOLDER, f)
            )
            count += 1
    print(f"✅ [PHASE 7] {count} valid files backed up to: {DRIVE_BACKUP_FOLDER}")

print("\n🎉 [PHASE 8] Pipeline complete. Your data is safe.")

[PHASE 0] Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.7 MB/s eta 0:00:00
[PHASE 0] Dependencies installed.
[PHASE 1] Authenticating with Earth Engine...
[PHASE 1] Earth Engine ready.
[PHASE 1] Mounting Google Drive...
Mounted at /content/drive
✅ [PHASE 1] Environment ready.

[PHASE 5] STARTING NaN-AWARE LSI PIPELINE (45 TILES)

[Tile 0] [PHASE 2] Selecting bands: ['S2_IOI_nd', 'S2_FII_nd', 'S2_CI_nd', 'L8_IOI_nd', 'L8_FII_nd', 'L8_CI_nd']
[Tile 0] [PHASE 2] Downloading 6-band subset (spatial tiling)...


/usr/local/lib/python3.12/dist-packages/geemap/common.py:12333: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)
/usr/local/lib/python3.12/dist-packages/geemap/common.py:12334: FutureWarning: 'num_threads' is deprecated and has no effect.  'max_requests' and 'max_cpus' can be used to limit concurrency.
  img.download(filename, overwrite=overwrite, num_threads=num_threads, **kwargs)


...zim/assets/tiles/feature_stack_tile_0:   0%|          |0/2 tiles [00:00<?]

/usr/local/lib/python3.12/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'projects/mineral-prospectivity-zim/assets/tiles/feature_stack_tile_0'.
  return STACClient().get(self.id)


[Tile 0] [PHASE 3] Download complete. Computing LSI maps...
[Tile 0] [PHASE 3]   band 1/6: S2_IOI_nd...
[Tile 0] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 0] [PHASE 3]     alpha stdDev: 0.94775, mean: 0.68193
[Tile 0] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_0_S2_IOI_nd.tif
[Tile 0] [PHASE 3]   band 2/6: S2_FII_nd...
[Tile 0] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 0] [PHASE 3]     alpha stdDev: 0.94754, mean: 0.68320
[Tile 0] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_0_S2_FII_nd.tif
[Tile 0] [PHASE 3]   band 3/6: S2_CI_nd...
[Tile 0] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 0] [PHASE 3]     alpha stdDev: 0.94731, mean: 0.68326
[Tile 0] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_0_S2_CI_nd.tif
[Tile 0] [PHASE 3]   band 4/6: L8_IOI_nd...
[Tile 0] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 0] [PHASE 3]     alpha stdDev: 0.94789, mean: 0.68152
[Tile 0] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_0_L8_IOI_nd.tif
[T

...

/usr/local/lib/python3.12/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'projects/mineral-prospectivity-zim/assets/tiles/feature_stack_tile_43'.
  return STACClient().get(self.id)


[Tile 43] [PHASE 3] Download complete. Computing LSI maps...
[Tile 43] [PHASE 3]   band 1/6: S2_IOI_nd...
[Tile 43] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 43] [PHASE 3]     alpha stdDev: 0.54351, mean: 0.16456
[Tile 43] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_43_S2_IOI_nd.tif
[Tile 43] [PHASE 3]   band 2/6: S2_FII_nd...
[Tile 43] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 43] [PHASE 3]     alpha stdDev: 0.56544, mean: 0.18625
[Tile 43] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_43_S2_FII_nd.tif
[Tile 43] [PHASE 3]   band 3/6: S2_CI_nd...
[Tile 43] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 43] [PHASE 3]     alpha stdDev: 0.54095, mean: 0.16198
[Tile 43] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_43_S2_CI_nd.tif
[Tile 43] [PHASE 3]   band 4/6: L8_IOI_nd...
[Tile 43] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 43] [PHASE 3]     alpha stdDev: 0.53720, mean: 0.15674
[Tile 43] [PHASE 3]   -> saved /content/lsi_outputs/lsi_til

...im/assets/tiles/feature_stack_tile_44:   0%|          |0/336 tiles [00:00<?]

/usr/local/lib/python3.12/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'projects/mineral-prospectivity-zim/assets/tiles/feature_stack_tile_44'.
  return STACClient().get(self.id)


[Tile 44] [PHASE 3] Download complete. Computing LSI maps...
[Tile 44] [PHASE 3]   band 1/6: S2_IOI_nd...
[Tile 44] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 44] [PHASE 3]     alpha stdDev: 0.84829, mean: 0.47124
[Tile 44] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_44_S2_IOI_nd.tif
[Tile 44] [PHASE 3]   band 2/6: S2_FII_nd...
[Tile 44] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 44] [PHASE 3]     alpha stdDev: 0.85099, mean: 0.48152
[Tile 44] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_44_S2_FII_nd.tif
[Tile 44] [PHASE 3]   band 3/6: S2_CI_nd...
[Tile 44] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 44] [PHASE 3]     alpha stdDev: 0.84973, mean: 0.47731
[Tile 44] [PHASE 3]   -> saved /content/lsi_outputs/lsi_tile_44_S2_CI_nd.tif
[Tile 44] [PHASE 3]   band 4/6: L8_IOI_nd...
[Tile 44] [PHASE 3]     NaN fraction in raw band: 0.000
[Tile 44] [PHASE 3]     alpha stdDev: 0.84802, mean: 0.46978
[Tile 44] [PHASE 3]   -> saved /content/lsi_outputs/lsi_til